# Travel Agents

Starts the three ADK-backed travel agents: Air Ticketing, Hotel Booking, and Car Rental.
Each connects to the MCP server for tool access (SQL queries against `travel_agency.db`).

In [1]:
import asyncio
import json
import logging
import os
import re
from collections.abc import AsyncIterable
from typing import Any

import uvicorn
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.genai import types as genai_types

import prompts
from common import AgentRunner, BaseAgent, build_a2a_app, get_mcp_server_config

load_dotenv()
logger = logging.getLogger(__name__)

/Users/paul/Documents/code/a2a-test/.venv/lib/python3.13/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [2]:
class TravelAgent(BaseAgent):
    """ADK-backed travel agent that connects to the MCP server for tools."""

    def __init__(self, agent_name: str, description: str, instructions: str):
        super().__init__(agent_name=agent_name, description=description, content_types=['text', 'text/plain'])
        self.instructions = instructions
        self.agent = None

    async def _init_agent(self):
        config = get_mcp_server_config()
        tools = await MCPToolset(connection_params=SseServerParams(url=config.url)).get_tools()
        model = os.getenv('LITELLM_MODEL', 'anthropic/claude-haiku-4-5-20251001')
        self.agent = Agent(
            name=self.agent_name, instruction=self.instructions,
            model=LiteLlm(model=model),
            disallow_transfer_to_parent=True, disallow_transfer_to_peers=True,
            generate_content_config=genai_types.GenerateContentConfig(temperature=0.0),
            tools=tools,
        )
        self.runner = AgentRunner()

    async def stream(self, query, context_id, task_id) -> AsyncIterable[dict[str, Any]]:
        if not query:
            raise ValueError('Query cannot be empty')
        if not self.agent:
            await self._init_agent()
        async for chunk in self.runner.run_stream(self.agent, query, context_id):
            if isinstance(chunk, dict) and chunk.get('type') == 'final_result':
                yield self._parse_response(chunk['response'])
            else:
                yield {'is_task_complete': False, 'require_user_input': False, 'content': f'{self.agent_name}: Processing...'}

    def _parse_response(self, raw):
        data = raw
        for pattern in [r'```\n(.*?)\n```', r'```json\s*(.*?)\s*```', r'```tool_outputs\s*(.*?)\s*```']:
            match = re.search(pattern, str(raw), re.DOTALL)
            if match:
                try:
                    data = json.loads(match.group(1))
                    break
                except json.JSONDecodeError:
                    data = match.group(1)
                    break
        if isinstance(data, dict):
            if data.get('status') == 'input_required':
                return {'response_type': 'text', 'is_task_complete': False, 'require_user_input': True, 'content': data['question']}
            return {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': data}
        try:
            data = json.loads(data)
            return {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': data}
        except Exception:
            return {'response_type': 'text', 'is_task_complete': True, 'require_user_input': False, 'content': data}

In [ ]:
agents = [
    (TravelAgent('AirTicketingAgent', 'Book air tickets', prompts.AIRFARE_COT_INSTRUCTIONS), 'agent_cards/air_ticketing_agent.json', 10103),
    (TravelAgent('HotelBookingAgent', 'Book hotels', prompts.HOTELS_COT_INSTRUCTIONS), 'agent_cards/hotel_booking_agent.json', 10104),
    (TravelAgent('CarRentalBookingAgent', 'Book rental cars', prompts.CARS_COT_INSTRUCTIONS), 'agent_cards/car_rental_agent.json', 10105),
]

async def serve_all():
    servers = []
    for agent, card_path, port in agents:
        app = build_a2a_app(agent, card_path)
        config = uvicorn.Config(app=app, host='localhost', port=port, log_level='info',)
        server = uvicorn.Server(config)
        servers.append(server)
        print(f'{agent.agent_name} running at http://localhost:{port}')
    await asyncio.gather(*(s.serve() for s in servers))

await serve_all()

AirTicketingAgent running at http://localhost:10103
HotelBookingAgent running at http://localhost:10104


INFO:     Started server process [5585]
INFO:     Waiting for application startup.
INFO:     Started server process [5585]
INFO:     Waiting for application startup.
INFO:     Started server process [5585]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Application startup complete.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10104 (Press CTRL+C to quit)
INFO:     Uvicorn running on http://localhost:10105 (Press CTRL+C to quit)
INFO:     Uvicorn running on http://localhost:10103 (Press CTRL+C to quit)


CarRentalBookingAgent running at http://localhost:10105
